# 04. Bảng và biểu đồ cho slide

Tạo bảng kết quả và các file PNG trong `outputs/results`. Biểu đồ nào thiếu dữ liệu sẽ được bỏ qua, nên có thể chạy lại sau mỗi lượt.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Đổi thành địa chỉ repo của bạn. Bỏ qua nếu đã giải nén repo vào /content hoặc thêm vào Kaggle.
REPO_URL = "https://github.com/TEN-TAI-KHOAN/vihsd-lora.git"

def find_repo():
    places = [Path.cwd(), *Path.cwd().parents, Path("/content/vihsd-lora"), Path("/kaggle/working/vihsd-lora")]
    places += list(Path("/kaggle/input").glob("*")) + list(Path("/kaggle/input").glob("*/vihsd-lora"))
    return next((place for place in places if (place / "src" / "vihsd_lora").is_dir()), None)

repo = find_repo()
if repo is None:
    repo = Path("/content" if Path("/content").exists() else "/kaggle/working") / "vihsd-lora"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
if not os.environ.get("VIHSD_SKIP_INSTALL"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
sys.path.insert(0, str(repo / "src"))
print("Repo:", repo)

In [ ]:
from vihsd_lora.config import Settings
from vihsd_lora.env import default_output_root, describe_gpu, detect_platform, load_secrets

print("Nền tảng:", detect_platform())
print("GPU:", describe_gpu())
print("Secret đã có:", load_secrets())

settings = Settings.from_env(output_root=default_output_root())
print("Kết quả lưu tại:", settings.output_root)

## Bảng kết quả

Gồm 13 dòng đã công bố cùng quy trình, mốc luôn đoán CLEAN và các lượt đã chấm test.

In [ ]:
from vihsd_lora.report import export_tables, main_table

display(main_table(settings))
print(export_tables(settings))

## Biểu đồ

In [ ]:
from IPython.display import Image, display as show
from vihsd_lora.plots import make_all

figures = make_all(settings)
for figure in figures:
    print(figure.name)
    show(Image(str(figure), width=760))

## Đưa biểu đồ lên W&B

Tạo run `slide-figures` để cả nhóm xem chung.

In [ ]:
from vihsd_lora.report import log_figures_to_wandb

log_figures_to_wandb(figures, settings)